In [1]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import time
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0

In [2]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = hidden_dim // n_head
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

32 language model layers, 32 heads, 128 head dim


In [3]:
img_dir = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    elif qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None

In [4]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
TYPES = ["attribute", "object", "relation"]
YES_ID, NO_ID = 3869, 1939

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
print(len(all_questions), "questions loaded")
print(processor.tokenizer.convert_ids_to_tokens([YES_ID, NO_ID]))

17492 questions loaded
['▁Yes', '▁No']


In [5]:
u = ln_f.weight.float().cpu() * (lm_head.weight[YES_ID] - lm_head.weight[NO_ID]).float().cpu()
_u_cache = {}


def u_on(device):
    if device not in _u_cache:
        _u_cache[device] = u.to(device)
    return _u_cache[device]


with torch.no_grad():
    head_proj = []
    for layer in layers:
        w = layer.self_attn.o_proj.weight
        head_proj.append(w.float().T.contiguous().view(n_head, head_dim, hidden_dim) @ u_on(w.device))

_dla_cache = {}


def _embed_hook(module, args, kwargs):
    h = args[0] if args else kwargs["hidden_states"]
    _dla_cache["embed"] = h[0, -1].float() @ u_on(h.device)


def _make_attn_hook(l):
    def hook(module, args):
        z = args[0][0, -1].float().reshape(n_head, head_dim)
        _dla_cache[f"attn_{l}"] = (z * head_proj[l]).sum(-1)
    return hook


def _make_mlp_hook(l):
    def hook(module, args, output):
        _dla_cache[f"mlp_{l}"] = output[0, -1].float() @ u_on(output.device)
    return hook


def _final_norm_hook(module, args):
    h = args[0][0, -1].float()
    _dla_cache["resid_dot"] = h @ u_on(h.device)
    _dla_cache["rms"] = torch.sqrt(h.pow(2).mean() + ln_f.variance_epsilon)


_hook_handles = [layers[0].register_forward_pre_hook(_embed_hook, with_kwargs=True)]
for l, layer in enumerate(layers):
    _hook_handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_make_attn_hook(l)))
    _hook_handles.append(layer.mlp.register_forward_hook(_make_mlp_hook(l)))
_hook_handles.append(ln_f.register_forward_pre_hook(_final_norm_hook))
print(len(_hook_handles), "hooks registered")

66 hooks registered


In [6]:
@torch.no_grad()
def dla_question(image_path, question_text):
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)

    _dla_cache.clear()
    outputs = model(
        input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
        attention_mask=inputs["attention_mask"],
    )
    last_logits = outputs.logits[0, -1].float().cpu()

    return {
        "embed": _dla_cache["embed"].cpu(),
        "attn": torch.stack([_dla_cache[f"attn_{l}"].cpu() for l in range(n_layer)]),
        "mlp": torch.stack([_dla_cache[f"mlp_{l}"].cpu() for l in range(n_layer)]),
        "resid_dot": _dla_cache["resid_dot"].cpu(),
        "rms": _dla_cache["rms"].cpu(),
        "logit_diff": last_logits[YES_ID] - last_logits[NO_ID],
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
    }

In [7]:
DLA_DIR = "/kaggle/working/dla_llava"
os.makedirs(DLA_DIR, exist_ok=True)

BATCH_SAVE_SIZE = 200
worst = {"rel_err_resid": 0.0, "err_logit": 0.0}


def collect(halluc_type):
    questions_subset = [q for q in all_questions if q["hallucination_type"] == halluc_type and q["answer_format"] == "yes_no"]
    paths = sorted(glob.glob(os.path.join(DLA_DIR, f"{halluc_type}_batch_*.pt")))
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path))
    todo = [q for q in questions_subset if q["question_id"] not in done_ids]
    print(f"{halluc_type}: {len(questions_subset)} questions, {len(done_ids)} already done, {len(todo)} to run")

    batch_entries, batch_index, skipped = [], len(paths), 0
    t_start = time.time()

    for i, q in enumerate(todo):
        image_path = resolve_image_path(q)
        if image_path is None or not os.path.exists(image_path):
            skipped += 1
            continue

        entry = dla_question(image_path, q["question_text"])
        total = entry["embed"] + entry["attn"].sum() + entry["mlp"].sum()
        scale = entry["embed"].abs() + entry["attn"].abs().sum() + entry["mlp"].abs().sum()
        worst["rel_err_resid"] = max(worst["rel_err_resid"], ((total - entry["resid_dot"]).abs() / scale).item())
        worst["err_logit"] = max(worst["err_logit"], (total / entry["rms"] - entry["logit_diff"]).abs().item())
        entry.update({k: q[k] for k in ("image_id", "question_id", "ground_truth_answer", "answer_format", "hallucination_type")})
        batch_entries.append(entry)

        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(DLA_DIR, f"{halluc_type}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1

        if i % 100 == 0 and i > 0:
            elapsed = time.time() - t_start
            avg = elapsed / (i + 1)
            remaining = avg * (len(todo) - i - 1)
            print(f"[{i+1}/{len(todo)}] {q['question_id']}: {entry['answer_text']!r} "
                  f"({avg:.2f}s/question avg, ETA {remaining/60:.1f} min, skipped: {skipped})")

    if batch_entries:
        torch.save(batch_entries, os.path.join(DLA_DIR, f"{halluc_type}_batch_{batch_index}.pt"))

    print(f"done -- {halluc_type}, {skipped} skipped, worst rel err resid {worst['rel_err_resid']:.5f}, "
          f"worst err logit {worst['err_logit']:.4f}, {(time.time()-t_start)/60:.1f} min")
    return skipped

In [8]:
skipped = collect("attribute")
assert skipped == 0, f"{skipped} questions skipped"

attribute: 7628 questions, 0 already done, 7628 to run
[101/7628] amber_1105: 'No' (0.47s/question avg, ETA 59.5 min, skipped: 0)
[201/7628] amber_1205: 'Yes' (0.49s/question avg, ETA 60.4 min, skipped: 0)
[301/7628] amber_1305: 'Yes' (0.51s/question avg, ETA 62.4 min, skipped: 0)
[401/7628] amber_1405: 'Yes' (0.52s/question avg, ETA 63.0 min, skipped: 0)
[501/7628] amber_1505: 'Yes' (0.53s/question avg, ETA 63.1 min, skipped: 0)
[601/7628] amber_1605: 'Yes' (0.54s/question avg, ETA 62.8 min, skipped: 0)
[701/7628] amber_1705: 'Yes' (0.54s/question avg, ETA 62.3 min, skipped: 0)
[801/7628] amber_1805: 'Yes' (0.54s/question avg, ETA 61.8 min, skipped: 0)
[901/7628] amber_1905: 'Yes' (0.54s/question avg, ETA 61.1 min, skipped: 0)
[1001/7628] amber_2005: 'Yes' (0.55s/question avg, ETA 60.4 min, skipped: 0)
[1101/7628] amber_2105: 'Yes' (0.55s/question avg, ETA 59.6 min, skipped: 0)
[1201/7628] amber_2205: 'Yes' (0.55s/question avg, ETA 58.8 min, skipped: 0)
[1301/7628] amber_2305: 'Yes' (

In [9]:
skipped = collect("object")
assert skipped == 0, f"{skipped} questions skipped"

object: 3000 questions, 0 already done, 3000 to run
[101/3000] pope_100: 'Yes' (0.57s/question avg, ETA 27.8 min, skipped: 0)
[201/3000] pope_200: 'Yes' (0.57s/question avg, ETA 26.7 min, skipped: 0)
[301/3000] pope_300: 'Yes' (0.57s/question avg, ETA 25.7 min, skipped: 0)
[401/3000] pope_400: 'Yes' (0.57s/question avg, ETA 24.8 min, skipped: 0)
[501/3000] pope_500: 'Yes' (0.57s/question avg, ETA 23.8 min, skipped: 0)
[601/3000] pope_600: 'Yes' (0.57s/question avg, ETA 22.9 min, skipped: 0)
[701/3000] pope_700: 'Yes' (0.57s/question avg, ETA 21.9 min, skipped: 0)
[801/3000] pope_800: 'No' (0.57s/question avg, ETA 21.0 min, skipped: 0)
[901/3000] pope_900: 'Yes' (0.57s/question avg, ETA 20.0 min, skipped: 0)
[1001/3000] pope_1000: 'Yes' (0.57s/question avg, ETA 19.0 min, skipped: 0)
[1101/3000] pope_1100: 'Yes' (0.57s/question avg, ETA 18.1 min, skipped: 0)
[1201/3000] pope_1200: 'Yes' (0.57s/question avg, ETA 17.1 min, skipped: 0)
[1301/3000] pope_1300: 'No' (0.57s/question avg, ETA 16

In [10]:
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
    if os.path.exists(zpath):
        os.remove(zpath)
print("Visual Genome downloaded, extracted, zips removed")

--2026-10-01 19:56:25--  https://cs.stanford.edu/people/rak248/VG_100K/images.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9730308001 (9.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K.zip’

/kaggle/working/VG_ 100%[===================>]   9.06G  42.0MB/s    in 6m 16s  

2026-10-01 20:02:41 (24.7 MB/s) - ‘/kaggle/working/VG_100K.zip’ saved [9730308001/9730308001]

--2026-10-01 20:02:41--  https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5471658058 (5.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K_2.zip’

/kaggle/working/VG_ 100%[===================>]   5.10G  19.4MB/s    in 2m 25s  

2026-10-01 20:05:06 (36.0 MB

In [11]:
skipped = collect("relation")
assert skipped == 0, f"{skipped} questions skipped"

relation: 6506 questions, 0 already done, 6506 to run
[101/6506] amber_13657: 'Yes' (0.57s/question avg, ETA 60.6 min, skipped: 0)
[201/6506] amber_13757: 'Yes' (0.56s/question avg, ETA 59.1 min, skipped: 0)
[301/6506] amber_13857: 'Yes' (0.58s/question avg, ETA 59.6 min, skipped: 0)
[401/6506] amber_13957: 'No' (0.57s/question avg, ETA 58.4 min, skipped: 0)
[501/6506] amber_14057: 'No' (0.57s/question avg, ETA 57.2 min, skipped: 0)
[601/6506] amber_14157: 'No' (0.57s/question avg, ETA 56.1 min, skipped: 0)
[701/6506] amber_14257: 'No' (0.57s/question avg, ETA 55.1 min, skipped: 0)
[801/6506] amber_14357: 'Yes' (0.57s/question avg, ETA 54.1 min, skipped: 0)
[901/6506] amber_14457: 'No' (0.57s/question avg, ETA 53.1 min, skipped: 0)
[1001/6506] amber_14557: 'No' (0.57s/question avg, ETA 52.2 min, skipped: 0)
[1101/6506] amber_14657: 'Yes' (0.57s/question avg, ETA 51.2 min, skipped: 0)
[1201/6506] amber_14757: 'Yes' (0.57s/question avg, ETA 50.2 min, skipped: 0)
[1301/6506] amber_14857: 

In [12]:
for t in TYPES:
    ids = []
    for path in glob.glob(os.path.join(DLA_DIR, f"{t}_batch_*.pt")):
        ids.extend(e["question_id"] for e in torch.load(path))
    expected = {q["question_id"] for q in all_questions if q["hallucination_type"] == t and q["answer_format"] == "yes_no"}
    print(f"{t:10s} entries {len(ids)} (expected {len(expected)}) | missing {len(expected - set(ids))} | duplicates {len(ids) - len(set(ids))}")
    assert set(ids) == expected and len(ids) == len(expected), f"{t} incomplete"

print(f"worst relative err vs final residual: {worst['rel_err_resid']:.5f}")
print(f"worst abs err vs model logit diff: {worst['err_logit']:.4f}")
assert worst["rel_err_resid"] < 0.05, "components do not sum to the final residual"
assert worst["err_logit"] < 0.5, "components do not reproduce the logit difference"
print("collection verified")

attribute  entries 7628 (expected 7628) | missing 0 | duplicates 0
object     entries 3000 (expected 3000) | missing 0 | duplicates 0
relation   entries 6506 (expected 6506) | missing 0 | duplicates 0
worst relative err vs final residual: 0.00018
worst abs err vs model logit diff: 0.0175
collection verified


In [13]:
with open(os.path.join(DLA_DIR, "dla_meta.json"), "w") as f:
    json.dump({
        "model": MODEL_NAME, "yes_id": YES_ID, "no_id": NO_ID, "n_layer": n_layer, "n_head": n_head,
        "worst_rel_err_resid": worst["rel_err_resid"], "worst_err_logit": worst["err_logit"],
    }, f)

with open(os.path.join(DLA_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "dla-llava",
        "id": f"{USERNAME}/dla-llava",
        "subtitle": "Direct logit attribution components for LLaVA yes/no questions",
        "description": "Per-question direct logit attribution for LLaVA-1.5-7B on every yes/no question in vlm-questions version 2. "
                       "embed, attn [layer, head], mlp [layer] and resid_dot are raw projections onto the Yes minus No direction "
                       "with the final norm gain folded in. Divide by rms to get logit units. logit_diff is the model's own Yes minus No logit.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

!kaggle datasets create -p {DLA_DIR}

Starting upload for file object_batch_11.pt
100%|██████████████████████████████████████| 1.13M/1.13M [00:00<00:00, 1.70MB/s]
Upload successful: object_batch_11.pt (1MB)
Starting upload for file relation_batch_12.pt
100%|██████████████████████████████████████| 1.13M/1.13M [00:00<00:00, 1.80MB/s]
Upload successful: relation_batch_12.pt (1MB)
Starting upload for file relation_batch_14.pt
100%|██████████████████████████████████████| 1.13M/1.13M [00:00<00:00, 1.83MB/s]
Upload successful: relation_batch_14.pt (1MB)
Starting upload for file attribute_batch_22.pt
100%|██████████████████████████████████████| 1.13M/1.13M [00:00<00:00, 1.88MB/s]
Upload successful: attribute_batch_22.pt (1MB)
Starting upload for file relation_batch_16.pt
100%|██████████████████████████████████████| 1.13M/1.13M [00:00<00:00, 1.76MB/s]
Upload successful: relation_batch_16.pt (1MB)
Starting upload for file relation_batch_18.pt
100%|██████████████████████████████████████| 1.13M/1.13M [00:00<00:00, 1.88MB/s]
Upload suc